# NjengaData — Analysis Pipeline
### Notebook 02 | Data Analyst: Mohamed Rashid

This notebook queries the cleaned SQLite database produced by the Data Engineering team.

**What happens here:**
- We connect to `njenga.db` — the single source of truth for all analysis
- We write SQL queries to answer four specific business questions
- Each query result is exported as a clean CSV for Stacey's visualization notebook

**Input:** `data/njenga.db` — produced by 01_data_cleaning.ipynb
**Output:** Four clean CSV files in `data/processed/`

> *"The database has the answers. SQL is how you ask the questions."*

---
## Section 1: Imports, Configuration and Database Connection

In [2]:
import sqlite3
import os
import pandas as pd

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:,.2f}'.format)

# ── Working directory fix ─────────────────────────────────────────
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# ── Paths ─────────────────────────────────────────────────────────
DB_PATH        = "data/njenga.db"
PROCESSED_PATH = "data/processed"

# Connect to the database — read-only mode
# read from the database, we never write to it
conn = sqlite3.connect(DB_PATH)

print(f"Connected to: {DB_PATH}")
print(f"Working directory: {os.getcwd()}")

# Confirm the tables are there
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table'",
    conn
)
print(f"\nTables available: {tables['name'].tolist()}")

ModuleNotFoundError: No module named 'pandas'

---
## Section 2: Business Questions

We answer four specific questions. Each question becomes one chart in Stacey's notebook.

| Query | Business Question | Chart it feeds |
|---|---|---|
| Q1 | What does a 2BR house in Nairobi cost, broken down by category? | Stacked bar chart |
| Q2 | How do 2BR costs compare across all four counties? | County comparison chart |
| Q3 | How have cement, steel and timber prices moved since 2019? | Material trend line |
| Q4 | Can a median household afford to build in their county? | Affordability gap chart |

### Query 1 — 2BR Nairobi Cost Breakdown
**Question:** Where does the money go when building a 2-bedroom house in Nairobi?

This feeds the stacked bar chart — the first visual judges will see.

In [ ]:
q1 = """
    SELECT
        cost_category,
        SUM(amount_kes)                                    AS total_kes,
        ROUND(
            SUM(amount_kes) * 100.0 / SUM(SUM(amount_kes)) OVER (),
            1
        )                                                  AS percentage
    FROM  cost_benchmarks
    WHERE county    = 'Nairobi'
      AND unit_type = '2BR'
      AND source    = 'CAHF'
    GROUP BY cost_category
    ORDER BY total_kes DESC
"""

df_q1 = pd.read_sql(q1, conn)

print("Q1 — Nairobi 2BR Cost Breakdown:")
print(df_q1.to_string(index=False))
print(f"\nTotal build cost: KES {df_q1['total_kes'].sum():,.0f}")

Q1 — Nairobi 2BR Cost Breakdown:
cost_category  total_kes  percentage
    Materials    1200000       36.40
         Land     850000       25.80
       Labour     480000       14.50
   Compliance     420000       12.70
    Overheads     350000       10.60

Total build cost: KES 3,300,000


### Query 2 — County Comparison (2BR)
**Question:** How does the cost of building a 2BR house differ across counties?

This feeds the county comparison chart.
We compare total build cost AND the cost breakdown structure side by side.

In [ ]:
q2 = """
    SELECT
        county,
        cost_category,
        SUM(amount_kes)  AS total_kes,
        ROUND(
            SUM(amount_kes) * 100.0 / SUM(SUM(amount_kes)) OVER (PARTITION BY county),
            1
        )                AS percentage
    FROM  cost_benchmarks
    WHERE unit_type = '2BR'
      AND source    = 'CAHF'
    GROUP BY county, cost_category
    ORDER BY county, total_kes DESC
"""

df_q2 = pd.read_sql(q2, conn)

# Pivot to show counties as columns — easier to read as a comparison table
q2_pivot = df_q2.pivot_table(
    index   = 'cost_category',
    columns = 'county',
    values  = 'total_kes',
    aggfunc = 'sum'
).reindex(['Land', 'Materials', 'Labour', 'Compliance', 'Overheads'])

# Add a total row
q2_pivot.loc['TOTAL'] = q2_pivot.sum()

print("Q2 — 2BR Cost Comparison by County (KES):")
print(q2_pivot.to_string())

Q2 — 2BR Cost Comparison by County (KES):
county          Kiambu  Mombasa  Nairobi   Nakuru
cost_category                                    
Land            580000   620000   850000   320000
Materials      1100000  1050000  1200000   980000
Labour          440000   420000   480000   392000
Compliance      280000   189000   420000   196000
Overheads       280000   294000   350000   245000
TOTAL          2680000  2573000  3300000  2133000


### Query 3 — Material Price Trends (2019–2024)
**Question:** How have cement, steel and timber prices moved over five years?

This feeds the trend line chart.
We use annual averages — one data point per material per year —
to smooth out quarterly noise and show the bigger directional story.

In [ ]:
q3 = """
    SELECT
        year,
        material,
        ROUND(AVG(price_index), 2)  AS avg_annual_index
    FROM  material_prices
    GROUP BY year, material
    ORDER BY material, year
"""

df_q3 = pd.read_sql(q3, conn)

# Pivot for readability — years as rows, materials as columns
q3_pivot = df_q3.pivot_table(
    index   = 'year',
    columns = 'material',
    values  = 'avg_annual_index'
)

print("Q3 — Annual Average Price Index by Material (Base: 2019 = 100):")
print(q3_pivot.to_string())

# Calculate total rise from 2019 to 2024 per material
print("\nPrice rise from 2019 to 2024:")
for material in q3_pivot.columns:
    base    = q3_pivot.loc[2019, material]
    current = q3_pivot.loc[2024, material]
    rise    = round(((current - base) / base) * 100, 1)
    print(f"  {material:<10} {rise:>6}% increase")

Q3 — Annual Average Price Index by Material (Base: 2019 = 100):
material  cement  steel  timber
year                           
2019      101.70 102.38  101.08
2020      104.88 102.20  103.38
2021      109.45 114.28  106.92
2022      117.20 126.08  112.42
2023      121.95 121.95  116.92
2024      125.13 122.42  120.08

Price rise from 2019 to 2024:
  cement       23.0% increase
  steel        19.6% increase
  timber       18.8% increase


### Query 4 — Affordability Gap
**Question:** Can a median household in each county afford to build a 2BR house?

This feeds the affordability gap chart — the most human chart in the dashboard.
We compare the minimum build cost per county against median annual household income.
The gap between the two lines is the housing crisis made visible.

In [ ]:
q4 = """
    SELECT
        cb.county,
        SUM(cb.amount_kes)              AS min_build_cost_kes,
        hi.median_annual_income_kes,
        hi.median_monthly_income_kes,
        ROUND(
            SUM(cb.amount_kes) * 1.0 /
            hi.median_annual_income_kes,
            1
        )                               AS years_of_income,
        ROUND(
            SUM(cb.amount_kes) * 1.0 /
            hi.median_monthly_income_kes,
            1
        )                               AS months_of_income
    FROM  cost_benchmarks cb
    JOIN  household_income hi
      ON  cb.county = hi.county
    WHERE cb.unit_type = '2BR'
      AND cb.source    = 'CAHF'
    GROUP BY cb.county, hi.median_annual_income_kes, hi.median_monthly_income_kes
    ORDER BY min_build_cost_kes DESC
"""

df_q4 = pd.read_sql(q4, conn)

print("Q4 — Affordability Gap by County:")
print(df_q4.to_string(index=False))

Q4 — Affordability Gap by County:
 county  min_build_cost_kes  median_annual_income_kes  median_monthly_income_kes  years_of_income  months_of_income
Nairobi             3300000                    720000                      60000             4.60             55.00
 Kiambu             2680000                    540000                      45000             5.00             59.60
Mombasa             2573000                    480000                      40000             5.40             64.30
 Nakuru             2133000                    420000                      35000             5.10             60.90


### Summary — Key Findings Before Export

Before writing files, we summarise the four findings in plain language.
This section becomes the Markdown narrative in the dashboard notebook.

In [ ]:
# Pull the headline numbers for the narrative summary
nairobi_2br   = df_q1['total_kes'].sum()
cheapest      = df_q2.groupby('county')['total_kes'].sum().idxmin()
cheapest_cost = df_q2.groupby('county')['total_kes'].sum().min()
gap           = nairobi_2br - cheapest_cost

def price_rise(material_name):
    mat = df_q3[df_q3['material'] == material_name].set_index('year')
    base    = mat.loc[2019, 'avg_annual_index']
    current = mat.loc[2024, 'avg_annual_index']
    return round(((current - base) / base) * 100, 1)

steel_rise  = price_rise('steel')
cement_rise = price_rise('cement')
timber_rise = price_rise('timber')

nairobi_years = df_q4[df_q4['county'] == 'Nairobi']['years_of_income'].values[0]
mombasa_years = df_q4[df_q4['county'] == 'Mombasa']['years_of_income'].values[0]

print("=" * 55)
print("  NJENGA DATA — KEY FINDINGS SUMMARY")
print("=" * 55)
print(f"\n  Finding 1 — Cost Breakdown")
print(f"  A 2BR house in Nairobi costs KES {nairobi_2br:,.0f}.")
print(f"  Land alone consumes 25.8% — before one brick is laid.")
print(f"  Compliance fees take 12.7% — pure regulatory cost.")

print(f"\n  Finding 2 — County Gap")
print(f"  {cheapest} is the most affordable county at KES {cheapest_cost:,.0f}.")
print(f"  That is KES {gap:,.0f} less than Nairobi for the same house.")

print(f"\n  Finding 3 — Materials Inflation")
print(f"  Cement has risen {cement_rise}% since 2019.")
print(f"  Steel has risen {steel_rise}% since 2019.")
print(f"  Contractors cannot stockpile cement — every pour costs more.")

print(f"\n  Finding 4 — Affordability Gap")
print(f"  A Nairobi household needs {nairobi_years} years of full income to build.")
print(f"  A Mombasa household needs {mombasa_years} years — the real affordability trap.")
print("=" * 55)

  NJENGA DATA — KEY FINDINGS SUMMARY

  Finding 1 — Cost Breakdown
  A 2BR house in Nairobi costs KES 3,300,000.
  Land alone consumes 25.8% — before one brick is laid.
  Compliance fees take 12.7% — pure regulatory cost.

  Finding 2 — County Gap
  Nakuru is the most affordable county at KES 2,133,000.
  That is KES 1,167,000 less than Nairobi for the same house.

  Finding 3 — Materials Inflation
  Cement has risen 23.0% since 2019.
  Steel has risen 19.6% since 2019.
  Contractors cannot stockpile cement — every pour costs more.

  Finding 4 — Affordability Gap
  A Nairobi household needs 4.6 years of full income to build.
  A Mombasa household needs 5.4 years — the real affordability trap.


---
## Section 3: Export Clean Data for Visualization

Each query result is saved as a CSV in data/processed/.
Stacey's notebook reads exclusively from these files — never from the database directly.

This separation means:
- Stacey can work independently once these files exist
- If the analysis changes, only these CSVs need updating — not the charts
- The processed files are the contract between Analyst and Viz Specialist

In [ ]:
# Ensure the processed folder exists
os.makedirs(PROCESSED_PATH, exist_ok=True)

# Define export paths
exports = {
    "q1_nairobi_2br_breakdown.csv" : df_q1,
    "q2_county_comparison.csv"     : df_q2,
    "q3_material_trends.csv"       : df_q3,
    "q4_affordability_gap.csv"     : df_q4,
}

# Export each dataframe
print("Exporting files to data/processed/:")
print("-" * 45)

for filename, dataframe in exports.items():
    filepath = os.path.join(PROCESSED_PATH, filename)
    dataframe.to_csv(filepath, index=False)
    size = os.path.getsize(filepath)
    print(f"  {filename:<40} {size:>6} bytes")

print("-" * 45)
print(f"\nAll {len(exports)} files exported successfully.")
print("\nHandoff to Stacey — open 03_dashboard.ipynb")

Exporting files to data/processed/:
---------------------------------------------
  q1_nairobi_2br_breakdown.csv                139 bytes
  q2_county_comparison.csv                    605 bytes
  q3_material_trends.csv                      364 bytes
  q4_affordability_gap.csv                    260 bytes
---------------------------------------------

All 4 files exported successfully.

Handoff to Stacey — open 03_dashboard.ipynb


---
## Section 4: Close Connection and Handoff Checklist

In [ ]:
# Close the database connection
conn.close()

print("Database connection closed.")


Database connection closed.
